# Notebook 01 — Setup and Folder Building
Run every cell top to bottom. This sets up your entire project structure.

## Cell 1 — Install all required packages

In [1]:
# Run this cell first — restart kernel after it finishes
!pip install torch torchvision
!pip install ultralytics
!pip install scikit-learn matplotlib seaborn
!pip install opencv-python pillow networkx
print("All packages installed!")


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


All packages installed!



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 2 — Import everything

In [2]:
import os, shutil, random
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.11.0+cpu


## Cell 3 — Build the YOLOv8 folder structure from your CVAT export

**IMPORTANT:** Before running this cell, copy your CVAT export `obj_train_data/` folder into your project folder.

Your project folder should look like:
```
campus_navigation_project/
    obj_train_data/          ← paste your CVAT export folder here
        IMG20260323142735.txt
        IMG20260323142735.jpg  ← also copy your photos here
        ...
```

In [3]:
import os, shutil, random

# ── STEP 1: Create all folders ────────────────────────────────
folders = [
    "data_yolo/images/train",
    "data_yolo/images/val",
    "data_yolo/labels/train",
    "data_yolo/labels/val",
    "data/train",
    "data/val",
    "data/test",
]
for f in folders:
    os.makedirs(f, exist_ok=True)
print("✓ All folders created")

# ── STEP 2: Get all image filenames from each subfolder in obj_train_data ─────
src = "obj_train_data"
all_files = []  # List of (image_path, label_path) tuples

# Iterate through each location subfolder
for location_folder in os.listdir(src):
    location_path = os.path.join(src, location_folder)
    
    # Skip if it's not a directory
    if not os.path.isdir(location_path):
        continue
    
    print(f"Scanning {location_folder}...")
    
    # Find all images in this location folder
    for f in os.listdir(location_path):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(location_path, f)
            # Check for matching .txt annotation
            label_name = os.path.splitext(f)[0] + ".txt"
            label_path = os.path.join(location_path, label_name)
            all_files.append((img_path, label_path))

print(f"✓ Found {len(all_files)} images across all subfolders in obj_train_data/")

if len(all_files) == 0:
    print("")
    print("⚠ No images found! Please ensure:")
    print("  1. Your JPG/PNG files are in the location subfolders")
    print("  2. Your .txt annotation files are in the same location subfolders")
else:
    # ── STEP 3: Shuffle and split 80% train / 20% val ────────
    random.seed(42)
    random.shuffle(all_files)
    split = int(len(all_files) * 0.8)
    train_files = all_files[:split]
    val_files   = all_files[split:]
    print(f"✓ Train: {len(train_files)} images | Val: {len(val_files)} images")

    # ── STEP 4: Copy images + labels to correct folders ──────
    def copy_yolo(files, img_dst, lbl_dst):
        for img_src, label_src in files:
            img_name = os.path.basename(img_src)
            # Copy image
            shutil.copy(img_src, os.path.join(img_dst, img_name))
            # Copy matching label if it exists
            if os.path.exists(label_src):
                label_name = os.path.basename(label_src)
                shutil.copy(label_src, os.path.join(lbl_dst, label_name))
            else:
                # Create empty label file for images with no signs
                label_name = os.path.splitext(img_name)[0] + ".txt"
                open(os.path.join(lbl_dst, label_name), "w").close()

    copy_yolo(train_files, "data_yolo/images/train", "data_yolo/labels/train")
    copy_yolo(val_files,   "data_yolo/images/val",   "data_yolo/labels/val")
    print("✓ All images and labels copied to data_yolo/")

✓ All folders created
Scanning bartels hall...
Scanning bergami hall...
Scanning dodd hall...
Scanning john and leona hall...
Scanning kplan hall...
Scanning maxy hall...
✓ Found 139 images across all subfolders in obj_train_data/
✓ Train: 111 images | Val: 28 images
✓ All images and labels copied to data_yolo/


## Cell 4 — Create data.yaml for YOLOv8 training

In [4]:
yaml_content = """path: data_yolo
train: images/train
val:   images/val

nc: 2
names:
  0: building_sign
  1: room_number
"""

with open("data_yolo/data.yaml", "w") as f:
    f.write(yaml_content)
print("✓ data.yaml created")
print(yaml_content)

✓ data.yaml created
path: data_yolo
train: images/train
val:   images/val

nc: 2
names:
  0: building_sign
  1: room_number



## Cell 5 — Verify your YOLO dataset is correct

In [5]:
train_imgs   = os.listdir("data_yolo/images/train")
train_labels = os.listdir("data_yolo/labels/train")
val_imgs     = os.listdir("data_yolo/images/val")
val_labels   = os.listdir("data_yolo/labels/val")

print(f"Train images : {len(train_imgs)}")
print(f"Train labels : {len(train_labels)}")
print(f"Val images   : {len(val_imgs)}")
print(f"Val labels   : {len(val_labels)}")

# Check matching
missing = []
for img in train_imgs:
    lbl = os.path.splitext(img)[0] + ".txt"
    if lbl not in train_labels:
        missing.append(img)

if missing:
    print(f"\n⚠ {len(missing)} images missing labels: {missing[:3]}")
else:
    print("\n✓ Every image has a matching label — ready to train YOLOv8!")

# Show label class counts
from collections import Counter
counts = Counter()
for lbl in os.listdir("data_yolo/labels/train"):
    path = f"data_yolo/labels/train/{lbl}"
    with open(path) as f:
        for line in f:
            if line.strip():
                cls = int(line.strip().split()[0])
                counts[cls] += 1

print(f"\nAnnotation counts in train set:")
print(f"  Class 0 (building_sign) : {counts[0]} boxes")
print(f"  Class 1 (room_number)   : {counts[1]} boxes")

Train images : 111
Train labels : 111
Val images   : 28
Val labels   : 28

✓ Every image has a matching label — ready to train YOLOv8!

Annotation counts in train set:
  Class 0 (building_sign) : 17 boxes
  Class 1 (room_number)   : 57 boxes


## Cell 6 — Setup ResNet50 folder structure

For ResNet50 you organise photos by location into named folders. Each folder name = one class.

In [6]:
# Show the expected structure
print("For ResNet50, organise YOUR photos into this structure:")
print()
print("data/")
print("  train/")
print("    dodds_exterior/        ← 40+ photos of Dodds Hall outside")
print("    kaplan_exterior/       ← 40+ photos of Kaplan Hall outside")
print("    library_exterior/      ← 40+ photos of Library outside")
print("    kaplan_floor2_room207/ ← 40+ photos of Room 207")
print("    kaplan_floor3_lab301/  ← 40+ photos of Lab 301")
print("  val/   (same folders, 10 photos each)")
print("  test/  (same folders, 10 photos each)")
print()
print("The FOLDER NAME becomes the class label automatically.")
print("You do NOT need CVAT for this — just put photos in the right folder.")
print()

# Create example subfolders (add your own based on your campus)
example_classes = [
    "dodds_exterior",
    "kaplan_exterior",
    "library_exterior",
    "kaplan_exterior",
    "bartels_exterior",
    "bergami_exterior",
    "john_and_leona_exterior",
    "maxy_exterior"
]
for cls in example_classes:
    for split in ["train", "val", "test"]:
        os.makedirs(f"data/{split}/{cls}", exist_ok=True)

print("✓ Example ResNet50 folders created.")
print("  Copy your campus photos into these folders now.")
print("  Add more folders for more locations.")

For ResNet50, organise YOUR photos into this structure:

data/
  train/
    dodds_exterior/        ← 40+ photos of Dodds Hall outside
    kaplan_exterior/       ← 40+ photos of Kaplan Hall outside
    library_exterior/      ← 40+ photos of Library outside
    kaplan_floor2_room207/ ← 40+ photos of Room 207
    kaplan_floor3_lab301/  ← 40+ photos of Lab 301
  val/   (same folders, 10 photos each)
  test/  (same folders, 10 photos each)

The FOLDER NAME becomes the class label automatically.
You do NOT need CVAT for this — just put photos in the right folder.

✓ Example ResNet50 folders created.
  Copy your campus photos into these folders now.
  Add more folders for more locations.
